## ROOT correctly resolved to the repo root and all 12 URLs are clean.

In [2]:
from pathlib import Path
from dotenv import load_dotenv

# Find the project root — works whether the kernel starts in the repo root or in notebooks/
ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "corpus_urls.txt").exists())

# Load Azure creds from the project-root .env into the environment
load_dotenv(ROOT / ".env")

# Read the 12 URLs (skip blank lines and # comment lines)
urls = [
    ln.strip()
    for ln in (ROOT / "corpus_urls.txt").read_text().splitlines()
    if ln.strip() and not ln.strip().startswith("#")
]

print(f"ROOT = {ROOT}")
print(f"{len(urls)} URLs loaded")
urls

ROOT = C:\Demo\verified-rag-assistant
12 URLs loaded


['https://docs.langchain.com/oss/python/langchain/overview',
 'https://docs.langchain.com/oss/python/langchain/install',
 'https://docs.langchain.com/oss/python/langchain/quickstart',
 'https://docs.langchain.com/oss/python/langchain/models',
 'https://docs.langchain.com/oss/python/langchain/retrieval',
 'https://docs.langchain.com/oss/python/langchain/rag',
 'https://docs.langchain.com/oss/python/langchain/knowledge-base',
 'https://docs.langchain.com/oss/python/langchain/agents',
 'https://docs.langchain.com/oss/python/langgraph/overview',
 'https://docs.langchain.com/oss/python/langgraph/agentic-rag',
 'https://docs.langchain.com/oss/python/integrations/splitters',
 'https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter']

## load the pages and check what we actually got.

In [4]:
import os
from langchain_community.document_loaders import WebBaseLoader

# Polite, non-blank User-Agent (some sites reject blank ones; also silences a warning)
os.environ.setdefault("USER_AGENT", "verified-rag-assistant/0.1 (demo)")

loader = WebBaseLoader(urls) # give it all 12 URLs
docs = loader.load()         # fetch + parse each page into a Document

In [5]:
docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/overview', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-te

In [6]:
type(docs)

list

In [7]:
print(f"{len(docs)} documents loaded\n")

12 documents loaded



In [38]:
for d in docs:
    print(f"{len(d.page_content):>6} chars | {d.metadata.get('source')}")

  9237 chars | https://docs.langchain.com/oss/python/langchain/overview
  2314 chars | https://docs.langchain.com/oss/python/langchain/install
 23106 chars | https://docs.langchain.com/oss/python/langchain/quickstart
 49711 chars | https://docs.langchain.com/oss/python/langchain/models
 11268 chars | https://docs.langchain.com/oss/python/langchain/retrieval
 82703 chars | https://docs.langchain.com/oss/python/langchain/rag
 25991 chars | https://docs.langchain.com/oss/python/langchain/knowledge-base
 49419 chars | https://docs.langchain.com/oss/python/langchain/agents
  6109 chars | https://docs.langchain.com/oss/python/langgraph/overview
 17495 chars | https://docs.langchain.com/oss/python/langgraph/agentic-rag
  4627 chars | https://docs.langchain.com/oss/python/integrations/splitters
  4526 chars | https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter


## First we need to find which HTML element wraps the real content (so we can tell the loader to keep only that). Let's inspect one page's raw HTML.

In [53]:
import requests
from bs4 import BeautifulSoup

# Fetch ONE page's raw HTML (WebBaseLoader hid this from us; now we look directly)
raw_html_page_content = requests.get(urls[0], headers={"User-Agent": os.environ["USER_AGENT"]}).text
soup = BeautifulSoup(raw_html_page_content, "html.parser")

# Does the page use a semantic main-content container?
for name in ["main", "article"]:
    el = soup.find(name)
    if el:
        print(f"<{name}> found — {len(el.get_text(strip=True))} chars")
    else:
        print(f"<{name}> NOT found")

<main> found — 7453 chars
<article> NOT found


## re-load all 12 pages, keeping only `<main>`

In [ ]:
from bs4 import SoupStrainer



## chunk the docs

In [39]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,   # ~1000 characters per chunk
        chunk_overlap=150, # repeat 150 chars between neighbors so context isn't cut mid-idea
)

chunks = splitter.split_documents(docs)

print(f"{len(docs)} docs → {len(chunks)} chunks\n")
print(chunks[0].page_content[:400])      # peek at the first chunk's text
print("\n--- metadata ---")
print(chunks[0].metadata)                 # note it keeps the 'source' URL

12 docs → 404 chunks

LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry La

--- metadata ---
{'source': 'https://docs.langchain.com/oss/python/langchain/overview', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}


## embed all chunks + store in Chroma (persisted to disk)

In [40]:
from langchain_openai import AzureOpenAIEmbeddings
from langchain_chroma import Chroma

embeddings_model_client = AzureOpenAIEmbeddings(
    azure_deployment=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

PERSIST_DIR = (ROOT / "chroma_db")  # DB saved here so we don't re-embed every run

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model_client,
    persist_directory=PERSIST_DIR,
    collection_name="langchain_docs",
)

In [41]:
type(vectorstore)

langchain_chroma.vectorstores.Chroma

In [42]:
print(f"Stored {vectorstore._collection.count()} vectors in {PERSIST_DIR}")

Stored 808 vectors in C:\Demo\verified-rag-assistant\chroma_db


In [43]:
vectorstore._collection

Collection(name=langchain_docs)

## test the retrieval (does a search return the right chunks?)

In [44]:
query = "How do I split documents into chunks for retrieval?"

results = vectorstore.similarity_search(query, k=3)   # top 3 closest chunks

for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.metadata.get("source")}")
    print(f"{doc.page_content[:200].strip()}...\n")

[1] https://docs.langchain.com/oss/python/integrations/splitters
Text splitters break large docs into smaller chunks that will be retrievable individually and fit within model context window limit.
There are several strategies for splitting documents, each with its...

[2] https://docs.langchain.com/oss/python/integrations/splitters
Text splitters break large docs into smaller chunks that will be retrievable individually and fit within model context window limit.
There are several strategies for splitting documents, each with its...

[3] https://docs.langchain.com/oss/python/integrations/splitters
Available text splitters:

Split by tokens
Split by characters

​Document structure-based
Some documents have an inherent structure, such as HTML, Markdown, or JSON files. In these cases, it’s benefic...



## build the RAG chain and ask it something

In [45]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 1) chat model — temperature=0 for consistent, factual answers
llm = AzureChatOpenAI(
    azure_deployment=os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    temperature=0,
)

# 2) retriever — pulls the top-4 chunks for a question
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 3) prompt — forces grounding + citations + honest "I don't know"  (this is the faithfulness lever)
prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below.\n"
    "If the answer is not in the context, reply exactly: I don't know.\n"
    "Cite the source URL(s) ONLY when you actually answer — never cite when you don't know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

# 4) fold retrieved chunks (with their source URL) into one context string
def format_docs(docs):
    return "\n\n".join(f"[Source: {d.metadata.get("source")}]\n{d.page_content}" for d in docs)

# 5) the RAG function: retrieve -> build prompt -> ask the LLM
def ask(question):
    docs = retriever.invoke(question)
    message = prompt.format(context=format_docs(docs), question=question)
    return llm.invoke(message).content

In [46]:
print(ask("How do I split documents into chunks, and which splitter is recommended?"))

To split documents into chunks, you can use text splitters that break large documents into smaller, retrievable chunks that fit within the model context window limit. For most use cases, it is recommended to start with the RecursiveCharacterTextSplitter, as it provides a solid balance between keeping context intact and managing chunk size. This default strategy works well out of the box, and adjustments should only be made if fine-tuning is necessary for specific applications. 

[Source: https://docs.langchain.com/oss/python/integrations/splitters]


## test the "honest I don't know" path

In [47]:
print(ask("What is the capital of France?"))

I don't know.
